# Star Cluster Injection Pipeline — Rubin CoaddPsf + Batch Processing

## 1. Imports

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

pipeline_path = os.path.expanduser('~/WORK/INJECT/star-cluster-injection-pipeline')
if pipeline_path not in sys.path:
    sys.path.insert(0, pipeline_path)

try:
    import galsim
    HAS_GALSIM = True
    print('GalSim available (Gaussian fallback ready).')
except ImportError:
    HAS_GALSIM = False
    print('WARNING: GalSim not found. PSF fallback disabled.')

from lsst.daf.butler import Butler
from lsst.geom import Point2D

from src.config import InjectionConfig, ClusterConfig
from src.inject import inject_cluster, create_injection_catalog
from src.light_profiles import KingProfile, PlummerProfile, EFFProfile, SersicProfile, mag_to_flux
from src.retrieval import ClusterRetrieval

plt.style.use('tableau-colorblind10')
%matplotlib inline
print('All imports successful.')

## 2. Load deepCoadd

In [ ]:
butler  = Butler('dp02', collections='2.2i/runs/DP0.2')
data_id = {'tract': 3828, 'patch': 24, 'band': 'i'}
coadd   = butler.get('deepCoadd', dataId=data_id)

CUTOUT     = 1500
image_full = coadd.image.array.copy()
BASE_IMAGE = image_full[:CUTOUT, :CUTOUT].copy()   # keep clean copy for each iteration

PIXEL_SCALE = 0.2
ZERO_POINT  = 27.0

psf_obj    = coadd.getPsf()
bbox       = coadd.getBBox()
BBOX_X_MIN = bbox.getMinX()
BBOX_Y_MIN = bbox.getMinY()

print(f'Cutout shape : {BASE_IMAGE.shape}')
print(f'BBox offset  : ({BBOX_X_MIN}, {BBOX_Y_MIN})')

## 3. PSF Diagnostic

In [ ]:
N_GRID = 5
xs = np.linspace(BBOX_X_MIN + 100, bbox.getMaxX() - 100, N_GRID).astype(int)
ys = np.linspace(BBOX_Y_MIN + 100, bbox.getMaxY() - 100, N_GRID).astype(int)

grid_fwhm = np.full((N_GRID, N_GRID), np.nan)
for i, y in enumerate(ys):
    for j, x in enumerate(xs):
        try:
            shape = psf_obj.computeShape(Point2D(int(x), int(y)))
            grid_fwhm[i, j] = shape.getDeterminantRadius() * 2.355
        except Exception:
            pass

valid = grid_fwhm[~np.isnan(grid_fwhm)]
PSF_FWHM_FALLBACK = float(np.median(valid)) if len(valid) > 0 else 3.5

print(f'Median PSF FWHM : {PSF_FWHM_FALLBACK:.3f} px ({PSF_FWHM_FALLBACK * PIXEL_SCALE:.3f} arcsec)')
print(f'Spatial std     : {valid.std():.4f} px')

## 4. Helper: get Rubin PSF at position (with GalSim fallback)

In [ ]:
def get_rubin_psf(psf_obj, cutout_x, cutout_y, bbox,
                  pixel_scale=0.2, fallback_fwhm=None):
    """
    Return a GalSim PSF object at (cutout_x, cutout_y).
    Primary  : Rubin CoaddPsf via computeImage()  -> InterpolatedImage
    Fallback : galsim.Gaussian with fallback_fwhm (pixels)
    """
    focal_x = float(cutout_x) + bbox.getMinX()
    focal_y = float(cutout_y) + bbox.getMinY()
    focal_pos = Point2D(focal_x, focal_y)

    try:
        psf_img   = psf_obj.computeImage(focal_pos)
        psf_arr   = psf_img.array.astype(np.float64)
        psf_arr  /= psf_arr.sum()
        gs_img    = galsim.Image(psf_arr, scale=pixel_scale)
        psf_gs    = galsim.InterpolatedImage(gs_img, normalization='flux')
        shape     = psf_obj.computeShape(focal_pos)
        fwhm_px   = shape.getDeterminantRadius() * 2.355
        psf_source = 'rubin'
    except Exception as e:
        print(f'  PSF fallback at ({cutout_x},{cutout_y}): {e}')
        if not HAS_GALSIM or fallback_fwhm is None:
            return None, None, 'none'
        psf_gs    = galsim.Gaussian(fwhm=fallback_fwhm * pixel_scale)
        fwhm_px   = fallback_fwhm
        psf_source = 'gaussian_fallback'

    return psf_gs, fwhm_px, psf_source


def convolve_stamp_with_psf(stamp, psf_gs, pixel_scale=0.2):
    """Convolve a normalised 2D stamp with a GalSim PSF."""
    ny, nx  = stamp.shape
    gs_img  = galsim.Image(stamp.astype(np.float64), scale=pixel_scale)
    src_gs  = galsim.InterpolatedImage(gs_img, normalization='flux')
    conv    = galsim.Convolve([src_gs, psf_gs])
    result  = galsim.Image(nx, ny, scale=pixel_scale)
    conv.drawImage(image=result, method='no_pixel')
    return result.array.astype(np.float64)


print('PSF helpers defined.')

## 5. Batch Configuration

In [ ]:
# ---- total experiment budget -------------------------------------------------
N_TOTAL      = 10_000   # total clusters across all iterations
N_ITERATIONS = 10       # number of independent injection runs
N_PER_ITER   = N_TOTAL // N_ITERATIONS   # clusters per iteration (1000)

# ---- cluster parameter space -------------------------------------------------
MAG_RANGE    = (19.0, 25.0)   # AB mag in i-band
R_HALF_RANGE = (2.0,  30.0)   # pixels
PROFILE_TYPE = 'plummer'      # 'plummer', 'king', 'eff', 'sersic'
EDGE_BUFFER  = 60             # pixels
ADD_NOISE    = True

print(f'Batch plan: {N_ITERATIONS} iterations × {N_PER_ITER} clusters = {N_TOTAL} total')

## 6. Batch Injection Loop

In [ ]:
all_injection_info = []   # accumulate across all iterations
all_detection_info = []   # accumulate detections across all iterations

for iteration in tqdm(range(N_ITERATIONS), desc='Batch iterations'):

    # -- fresh copy of the base image for each iteration ----------------------
    working_image = BASE_IMAGE.copy().astype(np.float64)

    # -- generate catalog for this iteration ----------------------------------
    catalog = create_injection_catalog(
        n_clusters   = N_PER_ITER,
        image_shape  = working_image.shape,
        mag_range    = MAG_RANGE,
        r_half_range = R_HALF_RANGE,
        profile_type = PROFILE_TYPE,
        edge_buffer  = EDGE_BUFFER,
        seed         = iteration,   # reproducible per iteration
    )

    iter_info = []

    for entry in catalog:
        cx, cy = entry['x'], entry['y']
        mag    = entry['magnitude']
        r_half = entry['r_half']

        # -- build light profile ----------------------------------------------
        profile = PlummerProfile(r_half=r_half)
        flux    = mag_to_flux(mag, zero_point=ZERO_POINT)

        stamp_size = max(51, int(10 * r_half) | 1)
        stamp = profile.generate_2d((stamp_size, stamp_size)).astype(np.float64)
        s = stamp.sum()
        if s > 0:
            stamp /= s
        stamp *= flux

        # -- get PSF (Rubin primary, Gaussian fallback) -----------------------
        psf_gs, fwhm_px, psf_source = get_rubin_psf(
            psf_obj, cx, cy, bbox,
            pixel_scale   = PIXEL_SCALE,
            fallback_fwhm = PSF_FWHM_FALLBACK,
        )

        # -- convolve ---------------------------------------------------------
        if psf_gs is not None:
            stamp = convolve_stamp_with_psf(stamp, psf_gs, pixel_scale=PIXEL_SCALE)

        # -- optional Poisson noise -------------------------------------------
        if ADD_NOISE:
            noise_sigma = np.sqrt(np.clip(stamp, 0, None))
            stamp += np.random.normal(0.0, np.where(noise_sigma > 0, noise_sigma, 1e-10))

        # -- place stamp into image -------------------------------------------
        ny, nx   = working_image.shape
        half     = stamp_size // 2
        y0i = max(0, cy - half);  y1i = min(ny, cy + half + 1)
        x0i = max(0, cx - half);  x1i = min(nx, cx + half + 1)
        y0s = y0i - (cy - half);  y1s = y0s + (y1i - y0i)
        x0s = x0i - (cx - half);  x1s = x0s + (x1i - x0i)
        working_image[y0i:y1i, x0i:x1i] += stamp[y0s:y1s, x0s:x1s]

        iter_info.append({
            'iteration'  : iteration,
            'id'         : entry['id'],
            'x'          : cx,
            'y'          : cy,
            'magnitude'  : mag,
            'r_half'     : r_half,
            'flux'       : flux,
            'psf_source' : psf_source,
            'psf_fwhm_px': fwhm_px,
        })

    all_injection_info.extend(iter_info)

    # -------------------------------------------------------------------------
    # DETECTION STEP — plug in your detector here
    # detections should be a list of dicts with at least 'x', 'y' keys
    # -------------------------------------------------------------------------
    # Example placeholder:
    # detections = run_your_detector(working_image)
    # all_detection_info.extend(detections)
    # -------------------------------------------------------------------------

    print(f'  iter {iteration:02d}: {len(iter_info)} clusters injected '
          f'({sum(1 for i in iter_info if i["psf_source"]=="rubin")} rubin PSF, '
          f'{sum(1 for i in iter_info if i["psf_source"]=="gaussian_fallback")} fallback)')

# -- convert to DataFrames for easy analysis ----------------------------------
df_injected  = pd.DataFrame(all_injection_info)
print(f'\nTotal injected records : {len(df_injected)}')
print(df_injected.head())

## 7. Combined Analysis & Completeness Plots

In [ ]:
# ---- match detections to injections -----------------------------------------
# Replace this with your actual matching / retrieval logic
# Here we show the scaffold assuming df_detections has 'x','y' columns

# MATCH_RADIUS_PX = 5.0
# retrieval = ClusterRetrieval(match_radius=MATCH_RADIUS_PX)
# df_matched = retrieval.match(df_injected, df_detections)

# ---- PSF source summary -----------------------------------------------------
print('PSF source breakdown:')
print(df_injected['psf_source'].value_counts())
print(f'Median FWHM (Rubin PSF) : '
      f'{df_injected[df_injected.psf_source=="rubin"]["psf_fwhm_px"].median():.3f} px')

In [ ]:
# ---- completeness curve (fill in detected=True/False after matching) ---------
# df_injected['detected'] = ...  # set from your matching step

# Example once detected column exists:
# mag_bins = np.arange(MAG_RANGE[0], MAG_RANGE[1] + 0.5, 0.5)
# bin_centers = 0.5 * (mag_bins[:-1] + mag_bins[1:])
# completeness = []
# for lo, hi in zip(mag_bins[:-1], mag_bins[1:]):
#     mask = (df_injected.magnitude >= lo) & (df_injected.magnitude < hi)
#     n_inj = mask.sum()
#     n_det = df_injected.loc[mask, 'detected'].sum()
#     completeness.append(n_det / n_inj if n_inj > 0 else np.nan)
#
# fig, ax = plt.subplots(figsize=(8, 5))
# ax.plot(bin_centers, completeness, 'o-')
# ax.set_xlabel('Injected magnitude')
# ax.set_ylabel('Completeness')
# ax.set_title(f'Combined completeness ({N_TOTAL} clusters, {N_ITERATIONS} iterations)')
# ax.set_ylim(0, 1.05)
# plt.tight_layout()
# plt.show()

# ---- magnitude distribution of all injected clusters -----------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_injected['magnitude'], bins=30, edgecolor='k')
axes[0].set_xlabel('Injected magnitude')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Magnitude distribution ({N_TOTAL} clusters)')

axes[1].hist(df_injected['r_half'], bins=30, edgecolor='k')
axes[1].set_xlabel('Half-light radius (px)')
axes[1].set_ylabel('Count')
axes[1].set_title('Half-light radius distribution')

plt.tight_layout()
plt.show()

## 8. Save Results

In [ ]:
out_path = os.path.expanduser('~/WORK/INJECT/results/batch_injection_catalog.csv')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_injected.to_csv(out_path, index=False)
print(f'Saved injection catalog to {out_path}')